# Security Classifier — Cross Verification

This notebook cross-checks the saved training summary, processed CSV files, model configuration, and final test metrics.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'Healthcare_Dataset_Preparation').exists():
    ROOT = ROOT.parents[1]

DATA_DIR = ROOT / 'Healthcare_Dataset_Preparation' / 'data' / 'processed'
OUTPUT_DIR = ROOT / 'Healthcare_Dataset_Preparation' / 'outputs'
SUMMARY_PATH = OUTPUT_DIR / 'security_classifier' / 'training_summary.txt'
METRICS_PATH = OUTPUT_DIR / 'evaluation' / 'test_metrics.txt'

print('Project root:', ROOT)
print('Training summary exists:', SUMMARY_PATH.exists())
print('Test metrics exists:', METRICS_PATH.exists())

Project root: C:\Users\raich\Desktop\llm\LLM-Security_Platform
Training summary exists: True
Test metrics exists: True


In [2]:
summary_text = SUMMARY_PATH.read_text(encoding='utf-8')
print(summary_text)

def extract_value(label, text):
    match = re.search(rf'{re.escape(label)}:\s*\n?([^\n]+)', text)
    return match.group(1).strip() if match else None

reported = {
    'Model': extract_value('Model', summary_text),
    'Maximum sequence length': extract_value('Maximum Sequence Length', summary_text),
    'Training samples': int(extract_value('Training Samples', summary_text)),
    'Validation samples': int(extract_value('Validation Samples', summary_text)),
    'Test samples': int(extract_value('Test Samples', summary_text)),
    'Number of classes': int(extract_value('Number of Classes', summary_text)),
    'Epochs': int(extract_value('Epochs', summary_text)),
    'Learning rate': extract_value('Learning Rate', summary_text),
    'Batch size': int(extract_value('Batch Size', summary_text)),
    'Best validation Macro F1': float(extract_value('Best Validation Macro F1', summary_text)),
    'Best validation accuracy': float(extract_value('Best Validation Accuracy', summary_text)),
}
pd.DataFrame(reported.items(), columns=['Parameter', 'Reported value'])


SECURITY CLASSIFIER TRAINING SUMMARY

Model:
distilbert-base-uncased

Maximum Sequence Length:
256

Training Samples:
10059

Validation Samples:
2306

Test Samples:
2306

Number of Classes:
5

Epochs:
4

Learning Rate:
2e-05

Batch Size:
16

Best Validation Macro F1:
0.8621

Best Validation Accuracy:
0.9328

Classes:
{0: 'jailbreak', 1: 'malicious', 2: 'phi', 3: 'safe', 4: 'suspicious'}




,Parameter,Reported value
0,Model,distilbert-base-uncased
1,Maximum sequence length,256
2,Training samples,10059
3,Validation samples,2306
4,Test samples,2306
5,Number of classes,5
6,Epochs,4
7,Learning rate,2e-05
8,Batch size,16
9,Best validation Macro F1,0.8621


In [3]:
files = {
    'train_processed': DATA_DIR / 'security_train_processed.csv',
    'validation_processed': DATA_DIR / 'security_validation_processed.csv',
    'test_processed': DATA_DIR / 'security_test_processed.csv',
}
datasets = {name: pd.read_csv(path) for name, path in files.items()}

verification = pd.DataFrame([
    {'Dataset': 'train_processed', 'Reported rows': reported['Training samples'], 'Actual CSV rows': len(datasets['train_processed'])},
    {'Dataset': 'validation_processed', 'Reported rows': reported['Validation samples'], 'Actual CSV rows': len(datasets['validation_processed'])},
    {'Dataset': 'test_processed', 'Reported rows': reported['Test samples'], 'Actual CSV rows': len(datasets['test_processed'])},
])
verification['Status'] = verification.apply(lambda row: 'PASS' if row['Reported rows'] == row['Actual CSV rows'] else 'MISMATCH', axis=1)
verification

,Dataset,Reported rows,Actual CSV rows,Status
0,train_processed,10059,10059,PASS
1,validation_processed,2306,2306,PASS
2,test_processed,2306,2306,PASS


In [4]:
train_df = datasets['train_processed']
print('Columns:', train_df.columns.tolist())
print('\nMissing values:')
display(train_df.isnull().sum().to_frame('missing_values'))
print('Duplicate prompts:', train_df['prompt'].duplicated().sum())

print('Class distribution:')
display(train_df['attack_type'].value_counts().rename_axis('attack_type').reset_index(name='count'))

print('Label mapping:')
display(train_df[['label', 'attack_type']].drop_duplicates().sort_values('label'))

assert train_df['attack_type'].nunique() == reported['Number of classes']
print('PASS: five-class security dataset verified.')

Columns: ['prompt', 'attack_type', 'severity', 'is_safe', 'phi_present', 'source_dataset', 'label', 'characters', 'words']

Missing values:


,missing_values
prompt,0
attack_type,0
severity,0
is_safe,0
phi_present,0
source_dataset,0
label,0
characters,0
words,0


Duplicate prompts: 0
Class distribution:


,attack_type,count
0,safe,5627
1,malicious,2874
2,phi,718
3,suspicious,661
4,jailbreak,179


Label mapping:


,label,attack_type
3,0,safe
7,1,malicious
1,2,phi
18,3,jailbreak
0,4,suspicious


PASS: five-class security dataset verified.


In [5]:
config_path = OUTPUT_DIR / 'security_classifier' / 'best_model' / 'config.json'
if config_path.exists():
    config = json.loads(config_path.read_text(encoding='utf-8'))
    print('Saved model type:', config.get('model_type'))
    print('Architecture:', config.get('architectures'))
    print('Number of labels:', config.get('num_labels'))
    print('Label mapping:', config.get('id2label'))
else:
    print('Model config not found:', config_path)

Saved model type: distilbert
Architecture: ['DistilBertForSequenceClassification']
Number of labels: None
Label mapping: {'0': 'jailbreak', '1': 'malicious', '2': 'phi', '3': 'safe', '4': 'suspicious'}


In [6]:
metrics_text = METRICS_PATH.read_text(encoding='utf-8')
print(metrics_text)

accuracy = float(re.search(r'Accuracy:\s*([0-9.]+)', metrics_text).group(1))
macro_f1 = float(re.search(r'Macro F1:\s*([0-9.]+)', metrics_text).group(1))
test_samples = int(re.search(r'Test samples:\s*(\d+)', metrics_text).group(1))

print(f'Final test samples: {test_samples}')
print(f'Final test accuracy: {accuracy:.2%}')
print(f'Final test Macro F1: {macro_f1:.2%}')

if test_samples != reported['Test samples']:
    print('NOTE: Test sample count differs from training_summary.txt; record the cleaned experiment split used for the final run.')

SECURITY CLASSIFIER TEST SET EVALUATION

Test samples: 2306
Accuracy: 0.9389
Macro Precision: 0.8640
Macro Recall: 0.8720
Macro F1: 0.8673
Weighted F1: 0.9401

Per-Class Metrics
--------------------------------------------------
safe: Precision=0.9759, Recall=0.9765, F1=0.9762
malicious: Precision=0.8852, Recall=0.8460, F1=0.8652
phi: Precision=1.0000, Recall=1.0000, F1=1.0000
jailbreak: Precision=1.0000, Recall=1.0000, F1=1.0000
suspicious: Precision=0.4587, Recall=0.5376, F1=0.4950

Final test samples: 2306
Final test accuracy: 93.89%
Final test Macro F1: 86.73%
